# The auto-anatomist's anneal, on a slider

Confidential, as the rest of bordercall is: keep this notebook, its copy in Colab and its outputs private.

The network learns to read the upper end-plate's width (Panjabi's EPWu) from a CT. In training a second branch also sees the vertebra's mask, and the two are mixed as (1 - w) times the CT's features plus w times the mask's. Over the run the mask goes down a ladder of three rungs as lambda rises from 0 to 1 (the vertebra's own identity goes, then its sharp edges, then the mask itself, which becomes a threshold of the CT), and then w falls from 1 to 0, so at the end the network reads the CT alone. The figure shows, every fifth step, the width the network measures on a held-out vertebra's raw CT beside the reference the oracle reads off that vertebra's mask, and on the same vertebra what the mask branch would be given, fading out as w falls.

It trains on phantoms, a demonstration and not a result. The notebook asks Colab for a T4 GPU when it opens (Runtime, Change runtime type shows what Colab gave). On the GPU it trains a network of the smoke run's widths; on a CPU, one of half those widths, which took 5.6 minutes on two CPU cores of a cloud container when this notebook was written.

## 1. Get the code

bordercall is private, and Colab's own GitHub sign-in, which opened this notebook, stays with Colab's page: the code in the notebook cannot use it. So the cell asks for your access itself, the first of these it finds:

- A token in Colab's Secrets, which makes this step automatic from then on: a fine-grained GitHub token whose resource owner is OpenSpineConsortium, with read-only access to the contents of bordercall alone, under the name `GITHUB_TOKEN` (the key on the left) with its notebook access on. Colab keeps it for your Google account, in every notebook. The cell reads it there and never prints it.
- Else, Sign in with GitHub: the cell shows a code and a green button. Press it: the code is copied and GitHub's page opens in a small window; paste the code, press Continue, then Authorize SymphonyBoard, and the cell goes on by itself. It clones with the token GitHub grants and keeps it in no output, no file and no variable.
- Else, a ZIP: on GitHub, open bordercall at the branch that holds this notebook (main once the demo has merged), press Code, then Download ZIP (about 90 MB), and upload it when the cell asks. The cell asks for it too when git cannot clone with the token, and says why.

The clone takes the demo's three folders alone (autoanatomist, scanimport and schwingcheck, about 3 MB), not bordercall's notes and records. The cell then installs the JAX, Flax and optax versions bordercall pins (requirements-autoanatomist.txt). On a GPU runtime it also installs the GPU build of that JAX for the CUDA version whose build Colab's image already carries (CUDA 13 on Colab's image of October 2026), so that one build alone drives the GPU, and it checks in a child process that JAX starts on the GPU: should it not start, the cell says why and the network trains on the CPU instead. Installing takes a minute or a few. This session never imports JAX: the training runs in a child process, which imports what pip installed, so nothing asks you to restart the session. Run in a clone of bordercall instead of Colab, the cell fetches and installs nothing, and stops at once if the clone's Python lacks plotly, which draws the figure.

In [ ]:
import html
import http.client
import importlib.metadata
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.parse
import urllib.request
import zipfile

REPOSITORY = "OpenSpineConsortium/bordercall"
REFS = ("main", "claude/project-thread-5hj0q5")   # the first that holds the demo is used; the second is its branch
BASE = "/content"                                  # Colab's working folder
CLONE = os.path.join(BASE, "bordercall")
NEEDED = ("autoanatomist", "scanimport", "schwingcheck")   # the demo's code; bordercall's notes and records stay on GitHub
PINNED = ("jax", "jaxlib", "flax", "optax")
MAJORS = ("13", "12")                              # the CUDA versions JAX's GPU builds are made for, newest first
APP_CLIENT_ID = "Iv23liTkIRov2FYN83wD"             # SymphonyBoard's, the App a reader signs in to; public, as every App's is
DEVICE_CODE = "https://github.com/login/device/code"
ACCESS_TOKEN = "https://github.com/login/oauth/access_token"


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def holds_demo(root):
    return os.path.exists(os.path.join(root, "autoanatomist", "demo.py"))


def token():
    """The GitHub token in Colab's Secrets, or None where there is none or the notebook may not read it."""
    from google.colab import userdata
    try:
        return userdata.get("GITHUB_TOKEN")
    except Exception:           # userdata's SecretNotFoundError or NotebookAccessError
        return None


def clone(secret, ref):
    """A shallow clone of ref with the demo's folders alone (git's sparse checkout, which leaves
    bordercall's notes and records, most of its megabytes, on GitHub), the token handed to git by a
    credential helper so that it is in no command line, no remote URL and no output. None where git
    cloned it; else git's last words."""
    helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
    git = ["git", "-c", "credential.helper=" + helper]
    env = dict(os.environ, GITHUB_TOKEN=secret, GIT_TERMINAL_PROMPT="0")
    shutil.rmtree(CLONE, ignore_errors=True)
    for words in (["clone", "--quiet", "--depth", "1", "--filter=blob:none", "--sparse", "--branch", ref,
                   "https://github.com/%s.git" % REPOSITORY, CLONE],
                  ["-C", CLONE, "sparse-checkout", "set", *NEEDED]):
        done = subprocess.run(git + words, env=env, capture_output=True, text=True)
        if done.returncode != 0:
            return (done.stderr.strip().splitlines() or ["git exited %d" % done.returncode])[-1]
    return None


def unpack_upload(why):
    """bordercall from a ZIP of it that GitHub made and the reader uploads."""
    from google.colab import files
    print("%s, so upload bordercall as a ZIP: on GitHub, the branch that holds this notebook, Code, Download ZIP." % why)
    name = next(iter(files.upload()))
    with zipfile.ZipFile(name) as archive:
        top = archive.namelist()[0].split("/")[0]
        archive.extractall(BASE)
    shutil.rmtree(CLONE, ignore_errors=True)
    os.rename(os.path.join(BASE, top), CLONE)
    if not holds_demo(CLONE):
        raise SystemExit("that ZIP holds no autoanatomist/demo.py: download the branch that holds this notebook")
    return CLONE


def json_object(answer):
    """An answer's body as a JSON object, or None where it is not one or cannot be read."""
    try:
        said = json.loads(answer.read())
    except (OSError, ValueError, http.client.HTTPException):
        return None
    return said if isinstance(said, dict) else None


def github(url, **fields):
    """One form post to GitHub's sign-in service and its answer; a refusal, an answer that is not
    JSON or a failed connection is an answer with an error, so the sign-in can say why it stopped."""
    body = urllib.parse.urlencode(dict(client_id=APP_CLIENT_ID, **fields)).encode()
    request = urllib.request.Request(url, data=body, headers={"Accept": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=30) as answer:
            said, status = json_object(answer), 200
    except urllib.error.HTTPError as refused:       # GitHub's refusals say why in JSON too
        said, status = json_object(refused), refused.code
    except (OSError, http.client.HTTPException) as failed:     # no connection, a timeout or a broken answer
        return {"error": str(failed)}
    return said if said is not None else {"error": "GitHub answered %d, not in JSON" % status}


SIGN_IN_BOX = """<div style="font-family:sans-serif;border:2px solid #2DA44E;border-radius:10px;padding:16px 20px;max-width:560px;margin:8px 0">
<div style="font-size:15px">Sign in with GitHub to get bordercall. Your code:</div>
<div style="font-size:30px;font-weight:bold;letter-spacing:3px;font-family:monospace;margin:8px 0 14px">%(shown)s</div>
<button id="%(id)s" style="font-size:15px;padding:10px 18px;background:#2DA44E;color:#FFFFFF;border:0;border-radius:6px;cursor:pointer">Copy the code and open GitHub</button>
<div style="font-size:13px;color:#555555;margin-top:10px">In GitHub's window: paste the code, Continue, then Authorize SymphonyBoard. This cell goes on by itself. No window? Open <a href="%(href)s" target="_blank">%(href)s</a>.</div>
</div>
<script>
document.getElementById(%(id_js)s).addEventListener("click", function () {
  var code = %(code_js)s, link = %(link_js)s;
  try { navigator.clipboard.writeText(code).catch(function () {}); } catch (e) {}
  if (!window.open(link, "github-sign-in", "popup,width=520,height=720")) { window.open(link, "_blank"); }
});
</script>"""


def js(text):
    """A string as JavaScript reads it inside a script element."""
    return json.dumps(text).replace("<", "\\u003c")


def show_code(link, code):
    """The sign-in's page and code: where the notebook shows HTML, the code large and a button that
    copies it and opens GitHub's page in a small window (a browser opens one only at a press), with
    the link beside it; and the words always."""
    print("Sign in with GitHub: open %s and enter the code %s" % (link, code), flush=True)
    try:
        from IPython.display import HTML, display
    except ImportError:
        return
    button = "sign-in-%d" % time.monotonic_ns()
    display(HTML(SIGN_IN_BOX % dict(id=button, id_js=js(button), shown=html.escape(code), code_js=js(code),
                                    link_js=js(link), href=html.escape(link, quote=True))))


def sign_in():
    """A token from GitHub's device flow: the reader opens GitHub's page, types the code shown and
    authorizes SymphonyBoard, and the cell clones with what GitHub grants, keeping it in no output,
    no file and no variable after the clone. None where the App does not offer the flow, or the
    reader declines or lets the code expire."""
    if not APP_CLIENT_ID:
        return None
    started = github(DEVICE_CODE)
    if "device_code" not in started:
        print("GitHub's sign-in is not open to this notebook (%s)." % started.get("error_description", started.get("error")))
        return None
    show_code(started["verification_uri"], started["user_code"])
    wait, deadline = started.get("interval", 5), time.monotonic() + started.get("expires_in", 900)
    while time.monotonic() < deadline:
        time.sleep(wait)
        answer = github(ACCESS_TOKEN, device_code=started["device_code"],
                        grant_type="urn:ietf:params:oauth:grant-type:device_code")
        if "access_token" in answer:
            print("Signed in.")
            return answer["access_token"]
        if answer.get("error") == "slow_down":
            wait = answer.get("interval", wait + 5)
        elif answer.get("error") != "authorization_pending":
            print("GitHub's sign-in stopped (%s)." % answer.get("error_description", answer.get("error")))
            return None
    print("The sign-in code expired.")
    return None


def fetch():
    """bordercall at the first of REFS that holds the demo, once per session: with the token in Colab's
    Secrets, else one from signing in with GitHub, else from a ZIP uploaded by hand, which is where a
    clone that fails at every ref ends too, saying why."""
    if holds_demo(CLONE):
        return CLONE
    secret = token() or sign_in()
    if secret is None:
        return unpack_upload("No GITHUB_TOKEN in Colab's Secrets and no sign-in")
    for ref in REFS:
        failed = clone(secret, ref)
        if failed is None and holds_demo(CLONE):
            return CLONE
        print("bordercall's %s: %s" % (ref, failed or "no autoanatomist/demo.py there"))
    return unpack_upload("Git could not get the demo with that token")


def repository_root():
    """Outside Colab: the clone this notebook sits in, the first folder up that holds autoanatomist/."""
    here = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(here, "autoanatomist")):
        if os.path.dirname(here) == here:
            raise SystemExit("run this notebook in a clone of bordercall, or in Colab")
        here = os.path.dirname(here)
    return here


def has_gpu():
    """Whether this session has an NVIDIA GPU, as Colab's GPU runtimes do."""
    try:
        listed = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
    except OSError:
        return False
    return listed.startswith("GPU")


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def plugins(major):
    """JAX's GPU build for one CUDA version: its plugin and the plugin's PJRT library."""
    return ("jax-cuda%s-plugin" % major, "jax-cuda%s-pjrt" % major)


def driver_major():
    """The newest CUDA version the GPU's driver runs, from nvidia-smi's header ("CUDA Version: 13.0"),
    or None where nvidia-smi says nothing of it."""
    try:
        header = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    except OSError:
        return None
    _, said, rest = header.partition("CUDA Version:")
    major = rest.strip().split(".")[0]
    return major if said and major.isdigit() else None


def cuda_major():
    """The CUDA version of the GPU build to install: the one whose build the image carries already
    (13 on Colab's GPU image of October 2026), so that pip replaces it and leaves no second plugin
    to take the GPU; else the newest the driver runs; else 12."""
    for major in MAJORS:
        if any(installed(name) for name in plugins(major)):
            return major
    driver = driver_major()
    for major in MAJORS:
        if driver is not None and int(driver) >= int(major):
            return major
    return "12"


def stale_plugins(major):
    """The image's GPU builds for the other CUDA versions: left beside the one installed, either
    could take the GPU, so pip takes them out."""
    return [name for other in MAJORS if other != major for name in plugins(other) if installed(name)]


def pins(root, cuda=None):
    """jax, jaxlib, flax and optax as bordercall pins them, and for a CUDA version the GPU build of
    that jaxlib, which JAX releases under jaxlib's version."""
    found = {}
    with open(os.path.join(root, "requirements-autoanatomist.txt")) as f:
        for line in f:
            name, _, version = line.split(";")[0].strip().partition("==")
            if name in PINNED and version:
                found[name] = version
    if cuda:
        found.update({name: found["jaxlib"] for name in plugins(cuda)})
    return found


def spec(name, version):
    """pip's words for one pin; a GPU plugin's extra brings NVIDIA's libraries, which pip leaves as
    they are where the image's meet it."""
    extra = "[with-cuda]" if name.startswith("jax-cuda") and name.endswith("-plugin") else ""
    return "%s%s==%s" % (name, extra, version)


def install(wanted, stale=()):
    """pip takes out the stale plugins and installs the pins the session lacks. This session never
    imports JAX, which trains in a child process (run_demo), so a package pip replaces here, numpy
    most often, needs no restart: the child imports the new one."""
    missing = [spec(name, version) for name, version in wanted.items() if installed(name) != version]
    stale = [name for name in stale if installed(name)]
    if missing and sys.version_info < (3, 12):
        raise SystemExit("bordercall's pinned JAX needs Python 3.12 or newer; this session has %d.%d" % sys.version_info[:2])
    if stale:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "--yes", "--quiet", *stale], check=True)
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)


def ensure_plotly():
    """plotly draws the figure. Colab has it; elsewhere it is the reader's to install, since bordercall's
    requirements do not pin it, and the cell says so before the minutes of training rather than after."""
    if importlib.util.find_spec("plotly") is not None:
        return
    if not in_colab():
        raise SystemExit("the figure needs plotly, which bordercall's requirements do not pin: "
                         "pip install plotly, then run this cell again")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "plotly"], check=True)


def why_no_gpu():
    """None where JAX starts on the GPU in a child process; else why not, in the child's last words.
    A GPU that JAX registers and then fails to start makes it raise at the first use, so this
    session asks a child first and imports JAX itself only after."""
    env = dict(os.environ, XLA_PYTHON_CLIENT_PREALLOCATE="false")
    env.pop("JAX_PLATFORMS", None)
    child = subprocess.run([sys.executable, "-c", "import jax; print(jax.default_backend())"], env=env,
                           capture_output=True, text=True)
    backend = child.stdout.split()[-1:]
    if child.returncode == 0 and backend == ["gpu"]:
        return None
    lines = [line.strip() for line in child.stderr.splitlines() if line.strip()]
    said = [line for line in lines if "Error" in line or "Warning" in line]
    return (said or lines or ["JAX chose its %s backend" % (backend or ["no"])[0]])[-1]


def settle_platform(cuda):
    """None where JAX will run on the GPU; else why not, with JAX_PLATFORMS set to the CPU before
    this session imports JAX."""
    why = why_no_gpu() if cuda else "this session has no GPU"
    if why:
        os.environ["JAX_PLATFORMS"] = "cpu"
    return why


def prepare():
    """bordercall's folder, the CUDA version of the GPU build in use (None without a GPU) and why JAX
    will not run on the GPU (None where it will), each step where it must come: the pins and the GPU
    build installed with the other builds taken out first, plotly asked for before the minutes of
    training, and JAX_PLATFORMS settled before the child that trains starts, which reads it then."""
    root = fetch() if in_colab() else repository_root()
    cuda = cuda_major() if has_gpu() else None
    if in_colab():
        install(pins(root, cuda), stale_plugins(cuda) if cuda else ())
    ensure_plotly()
    return root, cuda, settle_platform(cuda)


def run_demo(*words):
    """python -m autoanatomist.demo with these words, in a child process that imports the JAX pip
    installed, since this session never imports it; its progress printed as it comes."""
    said = []
    with subprocess.Popen([sys.executable, "-u", "-m", "autoanatomist.demo", *words], cwd=ROOT,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as child:
        for line in child.stdout:
            print(line, end="", flush=True)
            if line.strip():
                said.append(line.strip())
    if child.returncode != 0:
        raise SystemExit("the demo stopped with exit %d%s" % (child.returncode, ": " + said[-1] if said else ""))


def show(page):
    """A page the demo wrote, in the notebook."""
    from IPython.display import HTML, display
    with open(page) as f:
        display(HTML(f.read()))


ROOT, CUDA, WHY_CPU = prepare()
OUT = BASE if in_colab() else tempfile.mkdtemp(prefix="anneal_demo_")   # where the page and the kept run go
where = "an uploaded ZIP"
if os.path.exists(os.path.join(ROOT, ".git")):       # a folder in a clone, a file in a worktree
    where = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
print("bordercall at %s | jax %s, to train on the %s" % (where, installed("jax"), "GPU" if CUDA and not WHY_CPU else "CPU"))
if CUDA and WHY_CPU:
    print("This session has a GPU, but JAX did not start on it (%s), so it trains on the CPU, with the "
          "smaller network." % WHY_CPU)

## 2. Train, with a snapshot every fifth step

The smoke run's phantoms (8 short columns of three vertebrae to train on, 4 held out), its schedule, seeds and 150 steps, on the one measurement, with the naming layer off; the smoke run itself trains all seven measurements with the naming layer on. It trains in a child process, `python -m autoanatomist.demo`, which prints the width's mean error over the 12 held-out vertebrae every 25 steps, writes the figure as a page and keeps the run, so the next cell can draw another vertebra without training again. `WIDTHS` sets the network's size: on a GPU, 16,32,64, the smoke run's widths; on the CPU, 8,16,32. On two CPU cores of a cloud container with nothing else running, the whole run, making the phantoms included, took 5.6 minutes at (8, 16, 32) and 8.6 at (16, 32, 64); Colab's own CPUs and its T4 are not timed yet.

In [ ]:
WIDTHS = "16,32,64" if CUDA and not WHY_CPU else "8,16,32"
PAGE, KEPT = os.path.join(OUT, "anneal_demo.html"), os.path.join(OUT, "anneal_demo.pkl")
run_demo(PAGE, "--widths", WIDTHS, "--plotlyjs", "cdn", "--save", KEPT)

## 3. The figure

How to read it:

- Drag the slider under the figure, or press Play: each position is the network after that many steps.
- Left: one held-out vertebra's CT drawn as its bone surface at 200 HU: the one whose error at the end is the median of the 12, unless `VERTEBRA` names another. Orange is the end-plate width the network measures from the CT alone, blue the reference the oracle read off the mask, and the legend gives both lengths at that step. Green, on the same vertebra, is what the mask branch would be given in training at that step, drawn at its weight w: the vertebra's own mask, which takes in its neighbours, blurs and turns into a threshold of the CT as lambda rises, then fades out as w falls to 0. The network's reading never sees it. Drag to turn the vertebra, scroll to zoom.
- Top right: lambda (black) and w (green) over the run, the dots at the slider's step.
- Bottom right: the width on that vertebra over the run, the network's (orange) against the reference (blue, dashed), the same height as the schedule above it. The title gives both widths and the mean error over all 12 held-out vertebrae.

In [ ]:
VERTEBRA = None   # None: the one whose error at the end is the median of the 12; or any of 0 to 11
page = PAGE
if VERTEBRA is not None:      # drawn again from the kept run, with no training
    page = os.path.join(OUT, "anneal_demo_%d.html" % VERTEBRA)
    run_demo(page, "--load", KEPT, "--vertebra", str(VERTEBRA), "--plotlyjs", "cdn")
show(page)

## 4. Test on phantoms it never saw

The run kept its network. This cell draws new phantoms, none of them seen in training or held out during it, reads them from the CT alone with that network and plots the width it measures against the oracle's reference. There are five test sets of six columns of three vertebrae, each differing from the training phantoms in one way:

- new patients: new draws of the same kind of phantom, the in-distribution check;
- unseen levels: T6 to T10, where training saw T11 to L5 and the end plates are narrower;
- transitional spines: six-lumbar, T13, four-lumbar and lumbar-rib columns, at their transitions;
- noisier scanner: the new patients with 200 HU of noise, where training had 20;
- coarser scan: the new patients in 3 mm voxels, where training had 1.25 mm.

How to read it: left, each vertebra's width from the CT alone against the reference, on the dashed diagonal where the two agree; right, each set's mean error with its 95% interval over columns (a column is one phantom patient), beside always guessing the training phantoms' mean width, which reads no CT at all. A set where the network does no better than that guess is one it does not generalize to. The table under the figure gives the numbers. The cell takes a minute or so, most of it drawing the phantoms. The crops are still placed from the oracle's frame, as in training.

In [ ]:
TEST_PAGE = os.path.join(OUT, "anneal_test.html")
run_demo(TEST_PAGE, "--load", KEPT, "--test", "--plotlyjs", "cdn")
show(TEST_PAGE)

## What it is not

- Phantoms, not patients: each column is drawn from Panjabi's tables with 20 HU of noise, and a few minutes of training on 24 of their vertebrae show the anneal, not how well it reads a real CT. The test sets are phantoms too. Nothing is kept beyond the session.
- The network reads the CT alone, but its crop is placed as in training, from the oracle's frame turned and shifted at random. Placing the crop from the CT alone (scanimport's mask-free mode, `crops.from_found`) is not measured yet.
- The reference is the stand-in oracle's, which reads the phantoms' drawn values back within 0.44 mm on average and 1.34 mm at most (autoanatomist/README.md).